$$ \renewcommand{\sech}{\operatorname{sech}} $$

The [Korteweg–De Vries equation or KdV-equation](https://en.wikipedia.org/wiki/Korteweg%E2%80%93De_Vries_equation) is a non-linear PDE that models 1-spatial-dimension dispersive non-dissipative soliton (solitary wave) which has the form
$$
    \frac{\partial u}{\partial t} + 6u\frac{\partial u}{\partial x} + \frac{\partial^3 u}{\partial x^3} = 0
$$
which on the space-time domain of $(x,t) \in [-2,2] \times [-2,2]$, if we impose the boundary and initial condition 
- $u(x,-2) = -2\sech^2(x+8)$ for $x \in [-2,2]$
- $u(-2,t) = -2\sech^2(-2-4t)$ for $t \in [-2,2]$
- $u(2,t) = -\sech^2(2-4t)$ for $t \in [-2,2]$

then this leads to the closed-form solution
$$
    u(x,t) = -2\sech^2(x-4t)
$$

this notebook will primarily be on defining and training a PINN that can solve this KdV-equation and bechmark the predictions against the analytical solution above.

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.dont_write_bytecode = True

%load_ext autoreload
from model import KdVPINN

In [4]:
# Backend set up for neural networks
torch.cuda.empty_cache()
# Set seed
seed = 123
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_default_dtype(torch.float32)

# CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
def analytical_solution(x, t):
    return torch.sech(x-4*t)**2

In [6]:
"""
    PDE problem parameters
"""

# Domain
x_bounds = [-2,2]
t_bounds = [-2,2]

# Boundary and initial conditions
def boundary_left(t):
    return -2*torch.sech(-2-4*t)**2

def boundary_right(t):
    return -torch.sech(2-4*t)**2

def initial_condition(x):
    return -2*torch.sech(x + 8)**2

conditions = {
    "initial"        : initial_condition,
    "boundary_left"  : boundary_left,
    "boundary_right" : boundary_right
}

In [7]:
# Initialize solving PINN
%autoreload 2
pinn = KdVPINN(hidden_layers=[64,64,64,64,64]).to(device)